# Basic RAG Query Engine - Step by Step

[30-second quickstart](https://developers.llamaindex.ai/python/framework/#30-second-quickstart)

This project demonstrates how to build a **Retrieval-Augmented Generation (RAG) application** using vector search, help you understand and implement a basic RAG system step by step:

In [ ]:
pip install llama-index llama-index-llms-openai python-dotenv 

Install Langsmith SDK

In [ ]:
pip install -U openai langsmith

In [ ]:
from getpass import getpass
from dotenv import load_dotenv
from colorama import Fore, Style
import os

load_dotenv(override=True)  # Load environment variables from .env file

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  # Set OPENAI_API_KEY from environment variable
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded from environment variable.{Style.RESET_ALL}")

**LLM Setup**

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.llms.openai import OpenAI

embed_model = OpenAIEmbedding(
    api_key=os.getenv("OPENAI_API_KEY")
)
llm = OpenAI(model="gpt-4o-mini", temperature=0)

**Document Processing**

In [ ]:
# -----------------------------
# Download Data
# -----------------------------

!mkdir -p 'data/paul_graham/'
!wget 'https://raw.githubusercontent.com/run-llama/llama_index/main/docs/examples/data/paul_graham/paul_graham_essay.txt' -O 'data/paul_graham/paul_graham_essay.txt'

In [ ]:
# -----------------------------
# Data preparation and ingestion
# -----------------------------
documents = SimpleDirectoryReader("./data/paul_graham/").load_data()

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

# -----------------------------
# Documents splitting and transformation
# -----------------------------

text_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=10)
Settings.text_splitter = text_splitter

**Embeddings**
- Floating point numbers representing text. 
- The distance between two vectors measure their relatedness

In [ ]:
nodes = text_splitter.get_nodes_from_documents(documents)

for i, node in enumerate(nodes[:2]):  # just first 2
    embedding = embed_model.get_text_embedding(node.text)
    
    print(f"\n📄 Node {i}")
    print("Text:", node.text[:100])
    print("Embedding (first 10 values):", embedding[:10])

In [ ]:
# -----------------------------
# Indexing (creates embeddings automatically)
# -----------------------------
from llama_index.embeddings.openai import OpenAIEmbedding

index = VectorStoreIndex.from_documents(documents, transformations=[text_splitter], embed_model=embed_model)

**Customize Retrieval Process**

https://developers.llamaindex.ai/python/framework/module_guides/querying/retriever/

In [ ]:
from langsmith import traceable
from colorama import Fore, Style  

@traceable
def main():
    # -----------------------------
    #  Query processing 
    # -----------------------------
    user_input = "Who is Paul Graham."

    # -----------------------------
    #  Retrieval 
    # -----------------------------


    # -----------------------------
    #  Context Assembly + LLM Generation
    # -----------------------------

    # -----------------------------
    #  Outputs
    # -----------------------------
    for node in nodes:
        print(f"{Fore.BLUE}Retrieved Nodes:{Style.RESET_ALL}")
        print(node.get_content())

In [ ]:

if __name__ == "__main__":
    main()